## Figure 3 - unsmoothed MLS ClO and WACCM ClOx activation

Plot action: read the canonical chemistry product, apply display-only
selection and render this accepted logical figure as PNG and PDF.

Inputs: read `chemistry/clo_clox.nc`, the canonical product written by
diagnostic notebook 04. It contains Aura MLS daytime ClO, WACCM ClOx, and the
corresponding MERRA-2/WACCM minimum zonal-mean temperature fields over
60-82N, 1-100 hPa, on the 365-day October-September axis.

Processing: the plotting cell only validates and reshapes the stored
calendar-day anomalies. MLS uses the 15 complete 2004/05-2018/19 winters;
WACCM ClOx uses the independent 200-year archive. Event and climatology remain
unsmoothed.

Plot settings: panel (a) is MLS and panel (b) is WACCM. Both panels share one
-0.6 to 0.6 ppbv colorbar. Temperature context is limited to the green
`Tmin < 195 K` PSC I reference region and its dark-green 195 K boundary; it is
not a significance mask.

Outputs: `figure3_clo.png`
and the same-stem PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")

def draw_chemistry_product(
    relative: str,
    observed_variable: str,
    model_variable: str,
    observed_pressure: str,
    model_pressure: str,
    levels: np.ndarray,
    colorbar_ticks: np.ndarray,
    colorbar_label: str,
    figure_title: str,
    observed_title: str,
    model_title: str,
    stem: str,
    *,
    require_unsmoothed: bool = False,
) -> None:
    """Render the accepted two-panel chemistry figure from a canonical product."""
    variables = (
        observed_variable,
        model_variable,
        "merra2_tmin_k",
        "waccm_tmin_k",
    )
    product = load_dataset(relative, variables)
    if require_unsmoothed:
        smoothing = str(product.attrs.get("climatology_smoothing", "")).lower()
        method = str(product.attrs.get("method", "")).lower()
        if smoothing != "none" and "unsmoothed" not in method:
            raise ValueError("ClO/ClOx climatology must be unsmoothed")

    season_day = np.asarray(product["season_day"].values, dtype=float)
    if len(season_day) != 365:
        raise ValueError(f"{relative}: expected a 365-day no-leap Oct-Sep season")

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(15.8, 6.1),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    panels = (
        (
            axes[0],
            observed_variable,
            observed_pressure,
            "merra2_tmin_k",
            "merra2_pressure_hpa",
            observed_title,
        ),
        (
            axes[1],
            model_variable,
            model_pressure,
            "waccm_tmin_k",
            "waccm_t_pressure_hpa",
            model_title,
        ),
    )

    mappable = None
    for axis, variable, pressure_name, temp_name, temp_pressure, panel_title in panels:
        pressure = np.asarray(product[pressure_name].values, dtype=float)
        values = np.asarray(product[variable].values, dtype=float).T
        temperature_pressure = np.asarray(product[temp_pressure].values, dtype=float)
        temperature = np.asarray(product[temp_name].values, dtype=float).T

        filled = axis.contourf(
            season_day,
            pressure,
            values,
            levels=levels,
            cmap="RdBu_r",
            extend="both",
        )
        if mappable is None:
            mappable = filled

        cold_mask = np.where(
            np.isfinite(temperature),
            temperature < 195.0,
            np.nan,
        )
        axis.contourf(
            season_day,
            temperature_pressure,
            cold_mask.astype(float),
            levels=[0.5, 1.5],
            colors=["#66C2A5"],
            alpha=0.22,
            zorder=4,
        )
        axis.contour(
            season_day,
            temperature_pressure,
            temperature,
            levels=[195.0],
            colors=["#00796B"],
            linewidths=[1.4],
            linestyles=["solid"],
            zorder=5,
        )

        axis.set_yscale("log")
        axis.set_ylim(100, 1)
        axis.set_xlim(0, 364)
        axis.set_xticks(
            [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334],
            [
                "Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                "Apr", "May", "Jun", "Jul", "Aug", "Sep",
            ],
        )
        axis.set_xlabel("Month")
        axis.set_title(panel_title, fontsize=13)
        axis.grid(axis="x", color="0.87", linewidth=0.6)
        axis.tick_params(direction="out")

    axes[0].set_ylabel("Pressure (hPa)")
    colorbar = figure.colorbar(mappable, ax=axes, pad=0.02, aspect=32)
    colorbar.set_label(colorbar_label)
    colorbar.set_ticks(colorbar_ticks)
    axes[1].legend(
        handles=[
            Patch(
                facecolor="#66C2A5",
                edgecolor="#00796B",
                alpha=0.35,
                label="Tmin < 195 K (PSC I reference)",
            ),
            Line2D(
                [0],
                [0],
                color="#00796B",
                linewidth=1.4,
                linestyle="solid",
                label="Tmin = 195 K",
            ),
        ],
        loc="upper right",
        fontsize=8.0,
        frameon=True,
        framealpha=0.95,
    )
    figure.suptitle(figure_title, fontsize=14, y=1.035)
    save_figure(figure, stem)

draw_chemistry_product(
    "chemistry/clo_clox.nc",
    "mls_clo_anomaly_ppbv",
    "waccm_clox_anomaly_ppbv",
    "mls_pressure_hpa",
    "waccm_pressure_hpa",
    np.linspace(-0.6, 0.6, 41),
    np.linspace(-0.6, 0.6, 7),
    "ClO / ClO$_x$ anomaly (ppbv)",
    "Observed and modeled Arctic chlorine activation (60-82N)",
    "(a) Aura MLS daytime ClO anomaly, 2019/20",
    "(b) WACCM ClO$_x$ anomaly, year 0008",
    "figure3_clo",
    require_unsmoothed=True,
)


## Figure 3 - MLS and WACCM dehydration

Plot action: read the canonical chemistry product, apply display-only
selection and render this accepted logical figure as PNG and PDF.

Inputs: read `chemistry/h2o.nc`, the canonical product written by diagnostic
notebook 04. It contains Aura MLS and WACCM year-0008 H2O anomalies plus the
matching MERRA-2/WACCM temperature context over 60-82N, 1-100 hPa, on the
365-day October-September axis.

Processing: the plotting cell only validates and reshapes the stored
unsmoothed anomalies. MLS uses the 15 complete 2004/05-2018/19 winters; WACCM
H2O uses the complete non-target BWCN seasons, not the 200-year ClOx baseline.

Plot settings: panel (a) is MLS and panel (b) is WACCM. Both panels retain
the shared -1 to 1 ppmv colorbar. Temperature context is limited to the green
`Tmin < 195 K` PSC I reference region and its dark-green 195 K boundary; it is
not a significance mask.

Outputs: `figure3_h2o.png` and the
same-stem PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")

def draw_chemistry_product(
    relative: str,
    observed_variable: str,
    model_variable: str,
    observed_pressure: str,
    model_pressure: str,
    levels: np.ndarray,
    colorbar_ticks: np.ndarray,
    colorbar_label: str,
    figure_title: str,
    observed_title: str,
    model_title: str,
    stem: str,
    *,
    require_unsmoothed: bool = False,
) -> None:
    """Render the accepted two-panel chemistry figure from a canonical product."""
    variables = (
        observed_variable,
        model_variable,
        "merra2_tmin_k",
        "waccm_tmin_k",
    )
    product = load_dataset(relative, variables)
    if require_unsmoothed:
        smoothing = str(product.attrs.get("climatology_smoothing", "")).lower()
        method = str(product.attrs.get("method", "")).lower()
        if smoothing != "none" and "unsmoothed" not in method:
            raise ValueError("ClO/ClOx climatology must be unsmoothed")

    season_day = np.asarray(product["season_day"].values, dtype=float)
    if len(season_day) != 365:
        raise ValueError(f"{relative}: expected a 365-day no-leap Oct-Sep season")

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(15.8, 6.1),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    panels = (
        (
            axes[0],
            observed_variable,
            observed_pressure,
            "merra2_tmin_k",
            "merra2_pressure_hpa",
            observed_title,
        ),
        (
            axes[1],
            model_variable,
            model_pressure,
            "waccm_tmin_k",
            "waccm_t_pressure_hpa",
            model_title,
        ),
    )

    mappable = None
    for axis, variable, pressure_name, temp_name, temp_pressure, panel_title in panels:
        pressure = np.asarray(product[pressure_name].values, dtype=float)
        values = np.asarray(product[variable].values, dtype=float).T
        temperature_pressure = np.asarray(product[temp_pressure].values, dtype=float)
        temperature = np.asarray(product[temp_name].values, dtype=float).T

        filled = axis.contourf(
            season_day,
            pressure,
            values,
            levels=levels,
            cmap="RdBu_r",
            extend="both",
        )
        if mappable is None:
            mappable = filled

        cold_mask = np.where(
            np.isfinite(temperature),
            temperature < 195.0,
            np.nan,
        )
        axis.contourf(
            season_day,
            temperature_pressure,
            cold_mask.astype(float),
            levels=[0.5, 1.5],
            colors=["#66C2A5"],
            alpha=0.22,
            zorder=4,
        )
        axis.contour(
            season_day,
            temperature_pressure,
            temperature,
            levels=[195.0],
            colors=["#00796B"],
            linewidths=[1.4],
            linestyles=["solid"],
            zorder=5,
        )

        axis.set_yscale("log")
        axis.set_ylim(100, 1)
        axis.set_xlim(0, 364)
        axis.set_xticks(
            [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334],
            [
                "Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                "Apr", "May", "Jun", "Jul", "Aug", "Sep",
            ],
        )
        axis.set_xlabel("Month")
        axis.set_title(panel_title, fontsize=13)
        axis.grid(axis="x", color="0.87", linewidth=0.6)
        axis.tick_params(direction="out")

    axes[0].set_ylabel("Pressure (hPa)")
    colorbar = figure.colorbar(mappable, ax=axes, pad=0.02, aspect=32)
    colorbar.set_label(colorbar_label)
    colorbar.set_ticks(colorbar_ticks)
    axes[1].legend(
        handles=[
            Patch(
                facecolor="#66C2A5",
                edgecolor="#00796B",
                alpha=0.35,
                label="Tmin < 195 K (PSC I reference)",
            ),
            Line2D(
                [0],
                [0],
                color="#00796B",
                linewidth=1.4,
                linestyle="solid",
                label="Tmin = 195 K",
            ),
        ],
        loc="upper right",
        fontsize=8.0,
        frameon=True,
        framealpha=0.95,
    )
    figure.suptitle(figure_title, fontsize=14, y=1.035)
    save_figure(figure, stem)

draw_chemistry_product(
    "chemistry/h2o.nc",
    "mls_h2o_anomaly_ppmv",
    "waccm_h2o_anomaly_ppmv",
    "mls_pressure_hpa",
    "waccm_pressure_hpa",
    np.linspace(-1.0, 1.0, 41),
    np.linspace(-1.0, 1.0, 9),
    "H$_2$O anomaly (ppmv)",
    "Observed and modeled Arctic H$_2$O anomalies (60-82N)",
    "(a) Aura MLS H$_2$O anomaly, 2019/20",
    "(b) WACCM H$_2$O anomaly, year 0008",
    "figure3_h2o",
)
